autotunenet pilot 1 of 1

This notebook was generated by `benchmarks/cloud/kaggle.py` and runs part of an
AutoTuneNet benchmark experiment on Kaggle's GPU T4 x2.

- experiment: `benchmarks/experiments/pilot.yaml`
- repository commit: `a8bf99701f7253748179c9cebaa45fb1b19b271a`
- shards: 1/2, 2/2 (one per GPU, in parallel)

Steps: install the locked environment, find the attached datasets, run the
shards, and pack the results into `autotunenet_results.tar.gz`.

In [ ]:
import glob
import json
import os
import subprocess
import tarfile
import time

JOB = json.loads(r'''{"repo": "https://github.com/mitishraina/AutoTuneNet.git", "ref": "a8bf99701f7253748179c9cebaa45fb1b19b271a", "experiment": "benchmarks/experiments/pilot.yaml", "shards": ["1/2", "2/2"], "filters": []}''')
WORK = "/kaggle/temp/autotunenet"          # outside /kaggle/working, so not part of the output
REPO_DIR = os.path.join(WORK, "repo")
RESULTS = "/kaggle/working/results"
LOGS = "/kaggle/working/logs"
os.makedirs(WORK, exist_ok=True)
os.makedirs(LOGS, exist_ok=True)
START = time.time()


def sh(cmd, **kwargs):
    print(f"$ {cmd}", flush=True)
    subprocess.run(cmd, shell=True, check=True, **kwargs)


print(json.dumps(JOB, indent=2))

## 1. Environment
Clone the repository at the recorded commit and install the exact locked dependencies with uv.

In [ ]:
sh("pip install -q uv")
sh(f"git clone -q {JOB['repo']} {REPO_DIR}")
sh(f"git -C {REPO_DIR} checkout -q {JOB['ref']}")
sh("uv sync --locked --quiet", cwd=REPO_DIR)
sh("nvidia-smi --query-gpu=index,name,memory.total --format=csv")

## 2. Data
Use the attached `autotunenet-data` dataset if present (MNIST and CIFAR-10, wherever Kaggle unpacked them);
otherwise the benchmarks download the datasets on first use.

In [ ]:
def find_file(name, roots):
    for root in roots:
        for path in glob.glob(os.path.join(root, "**", name), recursive=True):
            return os.path.dirname(path)
    return None


DATA = os.path.join(WORK, "data")
os.makedirs(DATA, exist_ok=True)
extracted = os.path.join(WORK, "extracted")
for archive in glob.glob("/kaggle/input/**/*.tar", recursive=True):
    with tarfile.open(archive) as tar:
        tar.extractall(os.path.join(extracted, os.path.basename(archive)[:-4]), filter="data")
roots = ["/kaggle/input", extracted]

mnist_raw = find_file("train-images-idx3-ubyte", roots)
cifar_dir = find_file("data_batch_1", roots)
if mnist_raw:
    os.makedirs(os.path.join(DATA, "MNIST"), exist_ok=True)
    os.symlink(mnist_raw, os.path.join(DATA, "MNIST", "raw"))
if cifar_dir:
    os.symlink(cifar_dir, os.path.join(DATA, "cifar-10-batches-py"))
print(f"MNIST: {mnist_raw or 'will download'}\nCIFAR-10: {cifar_dir or 'will download'}")
ENV = {**os.environ, "AUTOTUNENET_DATA_DIR": DATA}

## 3. Run the shards
One `benchmarks.run` process per GPU. Each writes one JSON result per finished cell.

In [ ]:
procs = []
for gpu, shard in enumerate(JOB["shards"]):
    cmd = ["uv", "run", "python", "-m", "benchmarks.run", "--experiment", JOB["experiment"],
           "--shard", shard, "--device", "cuda", "--results-dir", RESULTS] + JOB["filters"]
    log_path = os.path.join(LOGS, f"shard_{shard.replace('/', '_of_')}.log")
    print(f"shard {shard} on GPU {gpu}: {' '.join(cmd)}", flush=True)
    procs.append((shard, log_path, subprocess.Popen(
        cmd, cwd=REPO_DIR, env={**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)},
        stdout=open(log_path, "w"), stderr=subprocess.STDOUT)))
failed = [shard for shard, _, p in procs if p.wait() != 0]
print(f"finished in {(time.time() - START) / 60:.1f} min; failed shards: {failed or 'none'}")

## 4. Package the results
Everything the launcher downloads is in `autotunenet_results.tar.gz`.

In [ ]:
with tarfile.open("/kaggle/working/autotunenet_results.tar.gz", "w:gz") as tar:
    if os.path.isdir(RESULTS):
        tar.add(RESULTS, arcname="results")
    tar.add(LOGS, arcname="logs")
for shard, log_path, _ in procs:
    print(f"--- last lines of shard {shard} ---")
    print("".join(open(log_path).readlines()[-15:]))